# Topic: Statistics: Standard Deviation & Bessel's Correction

## Definition (30-second explanation)
* Standard deviation (SD) measures how much individual data points deviate from the mean.
* A low SD means values are clustered tightly around the mean, while a high SD means values are spread out over a wide range.
* It is expressed in the exact same units as the original data, unlike variance which is squared.
* When calculating SD for a sample, we divide by (n-1) instead of N; this is known as Bessel's correction and it reduces bias in the estimation of the population.

## Why Interviewers Ask This
* To see if you understand the difference between a sample and a population, and how that impacts mathematical formulas in code.
* To ensure you don't just look at "averages" (means) when evaluating model performance or business metrics, but also consider volatility and risk.
* To test your knowledge of library defaults, as Python's `numpy` and `statistics` libraries handle degrees of freedom (`ddof`) differently.

## Core Concepts
* **Sample vs Population:** Use population SD (divide by N) when you have the entire dataset. Use sample SD (divide by n-1) when you are inferring about a population from a smaller subset.
* **Bessel's Correction (n-1):** Samples tend to underestimate the true variance of a population because they cluster around the sample mean. Dividing by (n-1) mathematically corrects this underestimation.
* **Volatility as Risk:** In finance or model predictions, a higher SD indicates higher volatility and less reliability.

## When to Use
* When checking the consistency, variability, or volatility of a dataset.
* When calculating Z-scores to detect outliers.
* When evaluating model performance variability across cross-validation folds.
* When comparing the risk profiles of different investments or A/B test effect sizes.

## Advantages
* Highly interpretable because it remains in the same physical units as the underlying data.
* It is a foundational component for advanced statistics, including confidence intervals and hypothesis testing (t-tests).

## Limitations
* SD is highly sensitive to extreme outliers because the deviations from the mean are squared before being summed.
* SD is not meaningful when the underlying data is heavily skewed; in such cases, the Interquartile Range (IQR) is a much better measure of spread.
* It cannot be applied to categorical variables.

## Common Comparisons
* **Standard Deviation vs. Variance:** Variance is the average squared deviation, meaning its units are squared (e.g., dollars squared). SD is the square root of variance, returning it to standard units (e.g., dollars).
* **Standard Deviation vs. IQR:** SD is best for normally distributed data. IQR is best for skewed data or data with heavy outliers.

## Common Interview Traps
* **The Library Default Trap:** Forgetting that `np.std()` defaults to `ddof=0` (population), while `statistics.stdev()` defaults to sample (n-1). 
* **The Variance Interpretation Trap:** Trying to explain variance in business terms. Always convert variance to standard deviation before presenting to stakeholders so the units make sense.
* **The Outlier Trap:** Blindly calculating SD on data with massive outliers without first checking the distribution.

## Python / SQL Syntax
```python
import numpy as np
import statistics

# NumPy defaults to Population SD (ddof=0). You MUST specify ddof=1 for Sample SD.
sample_sd_np = np.std(data, ddof=1)
pop_sd_np = np.std(data, ddof=0)

# Python's built-in statistics module defaults to Sample SD.
sample_sd_stat = statistics.stdev(data)
pop_sd_stat = statistics.pstdev(data)
```

## Important Formula
* **Sample Standard Deviation ($s$):** $\sqrt{\frac{\sum (x_i - \bar{x})^2}{n - 1}}$
* **Population Standard Deviation ($\sigma$):** $\sqrt{\frac{\sum (x_i - \mu)^2}{N}}$

## 45-Second Interview Answer
"Standard deviation measures the average distance of data points from their mean, indicating how clustered or spread out the data is. In an interview or production setting, my main focus is ensuring I use the correct degrees of freedom. If I'm analyzing a sample, I apply Bessel's correction by dividing by (n-1) to avoid underestimating the population's true variance. I also ensure I'm using the right tool—if the data is heavily skewed or has massive outliers, I'll switch from standard deviation to IQR because the squared terms in SD make it highly sensitive to anomalies."

## Example Question:

### Q1: 
**Scenario:**
"Two models have the same average accuracy of 85%. Model A has std dev = 2%, Model B has std dev = 10%. Which model would you choose for production and why?"

**Ideal Interview Answer:**
"I would confidently choose Model A. While both models yield the same average accuracy, standard deviation measures consistency. Model A's low standard deviation of 2% means its performance is tightly clustered, making it highly reliable in production. Model B is highly volatile—with a 10% standard deviation, its accuracy could swing to 95% on some days but crash to 75% on others. For production systems, reliability and predictability are almost always preferred over volatile peak performance."

**Common Mistakes Candidates Make:**
* Stating "they are the same since the average is the same," entirely missing the concept of variance/risk.
* Overcomplicating the answer with math formulas rather than explaining the business impact (reliability vs. unpredictability).

**One Likely Interviewer Follow-up:**
"Let's say we are evaluating cross-validation scores instead of daily production scores. How would you calculate this 2% standard deviation in Python using NumPy?" (Answer: `np.std(cv_scores, ddof=1)` because cross-validation folds represent a sample of all possible data).

## Practice Questions:

### Q1:
**Question 1 (SQL Coding):**
You are a Data Scientist analyzing manufacturing quality control. You need to flag "anomalous" product weights.

Write a SQL query that calculates the Sample Standard Deviation and the Mean of the product weights for each batch_id. Include only the batches where the sample standard deviation is greater than 2.5 (which indicates the batch is too volatile/inconsistent).

**Mock Schema**
```sql
CREATE TABLE quality_control (
    product_id INT,
    batch_id INT,
    weight_grams DECIMAL(10,2)
);

INSERT INTO quality_control (product_id, batch_id, weight_grams) VALUES
(1, 101, 50.5),
(2, 101, 51.0),
(3, 101, 50.8),
(4, 102, 45.0),
(5, 102, 55.0),
(6, 102, 50.0);
```

**Answer:**
```sql
SELECT
    batch_id,
    ROUND(AVG(weight_grams), 2) AS mean_weight,
    ROUND(STDDEV_SAMP(weight_grams), 2) AS sample_stddev
FROM quality_control
GROUP BY batch_id
HAVING STDDEV_SAMP(weight_grams) > 2.5;
```

**Common Mistakes Candidates Make:**
* **Using `WHERE` instead of `HAVING`:** Candidates often try to filter the aggregated standard deviation in the `WHERE` clause, which causes a syntax error since `WHERE` filters rows *before* aggregation.
* **Using the wrong function:** Using `STDDEV_POP()` when the data is clearly a sample of a continuous manufacturing process. 
* **Assuming dialect defaults:** Using just `STDDEV()` without knowing if the specific SQL dialect defaults to sample or population.

**One Likely Interviewer Follow-up:**
"If you wanted to find the exact *individual products* that are causing this high volatility, how would you change your query?" (Answer: I would switch from `GROUP BY` to Window Functions `OVER (PARTITION BY batch_id)` to calculate Z-scores for individual rows).

### Q2:
"Your manager wants to flag the specific individual products that are anomalies. Using the same quality_control table, write a SQL query to return all columns for products whose weight is more than 2 standard deviations away from their specific batch mean (i.e., absolute Z-score > 2). Treat the batch as a sample."

**Mock Schema**
```sql
CREATE TABLE quality_control (
    product_id INT,
    batch_id INT,
    weight_grams DECIMAL(10,2)
);

INSERT INTO quality_control (product_id, batch_id, weight_grams) VALUES
(1, 101, 50.5),
(2, 101, 51.0),
(3, 101, 50.8),
(4, 102, 45.0),
(5, 102, 55.0),
(6, 102, 50.0);
```

**Answer:**
```sql

-- Option 1:
WITH product_stats AS (
    SELECT 
        product_id,
        batch_id,
        weight_grams,
        AVG(weight_grams) OVER(PARTITION BY batch_id) AS batch_mean,
        STDDEV_SAMP(weight_grams) OVER(PARTITION BY batch_id) AS batch_stddev
    FROM quality_control
)
SELECT 
    product_id,
    batch_id,
    weight_grams,
    (weight_grams - batch_mean) / NULLIF(batch_stddev, 0) AS z_score
FROM product_stats
-- Use ABS() to catch both extremely heavy (>2) and extremely light (<-2) anomalies
WHERE ABS((weight_grams - batch_mean) / NULLIF(batch_stddev, 0)) > 2;

-- Option 2:
WITH metrics AS (
    SELECT
        batch_id,
        AVG(weight_grams) AS mean,
        STDDEV_SAMP(weight_grams) AS sample_stddev
    FROM quality_control
    GROUP BY batch_id
)
SELECT
    q.batch_id,
    q.product_id,
    q.weight_grams,
    (q.weight_grams - m.mean) / NULLIF(m.sample_stddev, 0) AS z_score
FROM quality_control q
JOIN metrics m
    ON q.batch_id = m.batch_id
WHERE ABS(
    (q.weight_grams - m.mean) / NULLIF(m.sample_stddev, 0)
) > 2;
```

**Common Mistakes Candidates Make:**
* **Using `GROUP BY` and `JOIN`:** While mathematically correct, joining a table to its own aggregated CTE is highly inefficient for large datasets compared to a single-pass Window Function.
* **Forgetting `ABS()`:** Outliers can be negative (too light) or positive (too heavy). If you just check `> 2`, you miss half the anomalies.
* **Division by Zero:** If a batch has only one item, its sample standard deviation is `NULL` or `0`. Trying to divide by `0` will crash the query in most SQL dialects. Wrapping the denominator in `NULLIF(batch_stddev, 0)` safely returns `NULL` instead of crashing.

**One Likely Interviewer Follow-up:**
"Why did you use a CTE here instead of putting the window functions directly in the `WHERE` clause?" (Answer: SQL execution order evaluates the `WHERE` clause before Window Functions. You cannot filter on a window function in the same query block it is defined; it must be wrapped in a CTE or Subquery).